<a href="https://colab.research.google.com/github/Harshada900/pyspark-practise/blob/main/16_understanding_parquet_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from pyspark.sql import SparkSession

# Initialize SparkSession if it doesn't already exist
spark = SparkSession.builder.appName("DataFrameCreation").getOrCreate()

# Task 1: Create two DataFrames with different schemas
df1 = spark.createDataFrame([("Raj", 25, 50000)], ["name", "age", "salary"])
df2 = spark.createDataFrame([("Priya", 28, 70000, "Engineering")],
                              ["name", "age", "salary", "department"])

In [4]:

# Write them to different folders
df1.write.mode("overwrite").parquet("/content/data/batch1")
df2.write.mode("overwrite").parquet("/content/data/batch2")

In [5]:
#Task 2 read without merge schema
df_no_merge = spark.read.parquet("/content/data/batch1", "/content/data/batch2")
df_no_merge.printSchema()
df_no_merge.show()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: long (nullable = true)

+-----+---+------+
| name|age|salary|
+-----+---+------+
|Priya| 28| 70000|
|  Raj| 25| 50000|
+-----+---+------+



In [6]:
#without mergeschema it only read schema from first table/file it finds!

In [7]:
# Task 3: Read WITH mergeSchema — observe the difference

df_merged = spark.read.option("mergeSchema", "true").parquet("/content/data/batch1", "/content/data/batch2")
df_merged.printSchema()
df_merged.show()



root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: long (nullable = true)
 |-- department: string (nullable = true)

+-----+---+------+-----------+
| name|age|salary| department|
+-----+---+------+-----------+
|Priya| 28| 70000|Engineering|
|  Raj| 25| 50000|       NULL|
+-----+---+------+-----------+



In [8]:
# now here department col appaers with null for batch1 rows/old rows !